Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from scraper import fetch_website_links, fetch_website_contents
import json
from IPython.display import Markdown, display, update_display

In [2]:
# Initialize and constants
# load_dotenv(override=True)
# api_key = os.getenv('OPENAI_API_KEY')
# openai = OpenAI()
OLLAMA_BASE_URL = "http://localhost:11434/v1"

model = "gemma:2b"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-

In [4]:
demo_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    demo_user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    demo_user_prompt += "\n".join(links)
    return demo_user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [7]:
def select_relevant_links(url):
    response = ollama.chat.completions.create(
        model = model,
        messages = [
            {"role":"system","content": demo_system_prompt},
            {"role":"user","content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links 

In [8]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'about page', 'url': 'https://edwarddonner.com/about-us'}]}

In [9]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {model}")
    response = ollama.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": demo_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [10]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemma:2b
Found 1 relevant links


{'links': [{'type': 'about page', 'url': 'https://huggingface.co/'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to call llm

In [11]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [12]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gemma:2b
Found 1 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
deepseek-ai/DeepSeek-V4-Pro
Updated
8 days ago
•
631k
•
3.55k
openai/privacy-filter
Updated
13 days ago
•
141k
•
1.28k
mistralai/Mistral-Medium-3.5-128B
Updated
1 day ago
•
15k
•
262
nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-BF16
Updated
about 2 hours ago
•
44.6k
•
233
poolside/Laguna-XS.2
Updated
2 days ago
•
12k
•
216
Browse 2M+ models
Spaces
Running
on
Zero
MCP
931
Wan2.2 14B Fast Preview
🐌
931
generate a video from an image with a text prompt
Running
on
Zero
MCP
2.54k
Wan2.2 14B Preview
🐌
2.54k
generate a video from an image w

In [13]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [14]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [15]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemma:2b
Found 1 relevant links


"\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\ndeepseek-ai/DeepSeek-V4-Pro\nUpdated\n8 days ago\n•\n631k\n•\n3.55k\nopenai/privacy-filter\nUpdated\n13 days ago\n•\n141k\n•\n1.28k\nmistralai/Mistral-Medium-3.5-128B\nUpdated\n1 day ago\n•\n15k\n•\n262\nnvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-BF16\nUpdated\nabout 2 hours ago\n•\n44.6k\n•\n234\npoolside/Laguna-XS.2\nUpdated\n2 days ago\n•\n12k\n•\n216\nBrowse 2M+ models\nSpaces\nRu

In [16]:
def create_brochure(company_name, url):
    response = ollama.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [17]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemma:2b
Found 1 relevant links


## Hugging Face: Your AI Journey to Success

**Building the Future of AI Together**

Welcome to Hugging Face, the **AI community building the future**. We empower the AI ecosystem by providing:

**Join the AI Movement:**

* Collaborate on **open-source models** and datasets for everyone to use.
* **Build your own AI applications** from scratch.
* **Share your work and build your reputation** with a global community.

**Here's what we offer:**

* **The AI Home for all your AI needs:** Explore our vast platform with models, datasets, spaces, and more.
* **Open source tools and resources:** Contribute to the future of AI with our developer portal and open-source projects.
* **Enterprise-grade solutions:** We provide secure and scalable platforms for large organizations.
* **Professional support:** Our dedicated team is here to help with any questions or challenges you encounter.

**We're passionate about AI and its potential to transform the world.** We invite you to join our vibrant community and start your AI journey with us.

**Explore the possibilities on Hugging Face.**

**Start your free trial today!**


**What We Do:**

* We build and maintain **high-quality AI models** across various domains.
* We offer **open-source tools and resources** to empower the AI community.
* We provide **custom solutions and services** to meet the specific needs of our clients.
* We are committed to fostering collaboration and sharing knowledge within the AI ecosystem.

**Join us, and let's create something amazing together!**

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [18]:
def stream_brochure(company_name, url):
    stream = ollama.chat.completions.create(
        model= model,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [19]:
create_brochure("neurohub", "https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gemma:2b
Found 2 relevant links


## NeuroHub: A Company Transforming Hiring with AI

**Welcome to NeuroHub, the AI-powered recruitment agency dedicated to empowering recruiters to find the most talented and engaging candidates.**

At NeuroHub, we understand that finding top talent is a competitive challenge. With the help of advanced AI technology, we aim to revolutionize the recruitment process and help businesses find their ideal fit.

**Our Mission:**

Our mission is to improve the efficiency, accuracy, and ethics of the hiring process. We achieve this by leveraging the latest AI techniques to predict, personalize, and optimize the candidate experience from start to finish.

**Our Expertise:**

Our team of experienced AI engineers and recruiters comprises industry veterans passionate about crafting impactful solutions that align with your company's goals. We are committed to staying at the forefront of AI advancements, ensuring we provide our clients with the most advanced technology at their disposal.

**What We Offer:**

* **AI-Powered Candidate Matching:** Our AI algorithms analyze massive datasets to identify patterns and predict future job requirements. This allows us to find qualified candidates who closely align with your company's needs.
* **Personalized Candidate Experiences:** We tailor each candidate's journey with a customized career path based on their skills, interests, and career goals. This ensures a frictionless experience and helps increase engagement.
* **Predictive Analytics and Insights:** By analyzing hiring data, we identify trends, predict future talent needs, and provide insights that help our clients optimize their recruitment strategies and make data-driven hiring decisions.
* **Enhanced Employer Branding:** Our AI tools help generate personalized brand awareness and reach, attracting top talent from diverse backgrounds and industries.

**Our Values:**

* **Integrity:** We operate with honesty and transparency in everything we do, building trust with our clients and partners.
* **Results:** We are relentlessly focused on delivering measurable results that meet our client's objectives.
* **Innovation:** We are passionate about staying ahead of the curve and embracing new AI innovations to optimize the hiring process.
* **Collaboration:** We work closely with our clients and partners to ensure their success and achieve their recruitment goals.

**Join the NeuroHub community and experience the difference of AI-powered recruitment.**

**Contact us today to learn more and explore how NeuroHub can help revolutionize your hiring process.**

**We look forward to assisting you on your journey to finding top talent!**